# Sparsity and Cluster Preserving QAT

## Install TensorFlow Model Optimization Toolkit
* 설치 완료 후 반드시 Runtime 재시작!
    * '런타임' > '세션 다시 시작' 메뉴 선택

In [ ]:
!pip install tensorflow-model-optimization

## Mount Google driver

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


In [ ]:
if colab :
  save_dir = '/content/drive/MyDrive/files/save/'
else :
  save_dir = '../files/save/'

## Import Module

In [ ]:
import tensorflow as tf
import numpy as np

import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

import tempfile

print(tf.__version__)
print(np.__version__)

2.19.0
1.26.4


## Load Dataset

In [ ]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

train_images = (train_images / 255.0).astype(np.float32)
test_images = (test_images / 255.0).astype(np.float32)

11490434/11490434 [==============================] - 0s 0us/step


In [ ]:
train_images = np.expand_dims(train_images, axis=-1)
test_images = np.expand_dims(test_images, axis=-1)

## Load Baseline Model for MNIST

In [ ]:
model = keras.models.load_model(save_dir + 'baseline_model.h5')
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0         
                                                                 
 dense (Dense)               (None, 128)               5

In [ ]:
_, baseline_model_accuracy = model.evaluate(
    test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)

Baseline test accuracy: 0.9904000163078308


## Pruning
* sparsity : 50%
* pruning schedule : Constant Sparsity

In [ ]:
pruning_params = {
    'pruning_schedule': tfmot.sparsity.keras.ConstantSparsity(0.5, begin_step=0, frequency=100)
}

callbacks = [
  tfmot.sparsity.keras.UpdatePruningStep()
]

pruned_model = tfmot.sparsity.keras.prune_low_magnitude(model, **pruning_params)

pruned_model.compile(
  loss=keras.losses.SparseCategoricalCrossentropy(),
  optimizer=keras.optimizers.Adam(learning_rate=1e-5),
  metrics=['accuracy'])

pruned_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_conv2d  (None, 26, 26, 32)        610       
  (PruneLowMagnitude)                                            
                                                                 
 prune_low_magnitude_max_po  (None, 13, 13, 32)        1         
 oling2d (PruneLowMagnitude                                      
 )                                                               
                                                                 
 prune_low_magnitude_conv2d  (None, 11, 11, 16)        9234      
 _1 (PruneLowMagnitude)                                          
                                                                 
 prune_low_magnitude_max_po  (None, 5, 5, 16)          1         
 oling2d_1 (PruneLowMagnitu                                      
 de)                                                    

## Fine tune the model for pruning

In [ ]:
pruned_model.fit(
  train_images,
  train_labels,
  epochs=3,
  validation_split=0.1,
  callbacks=callbacks)

Epoch 1/3
1688/1688 [==============================] - 18s 7ms/step - loss: 0.0191 - accuracy: 0.9947 - val_loss: 0.0393 - val_accuracy: 0.9895
Epoch 2/3
1688/1688 [==============================] - 11s 7ms/step - loss: 0.0118 - accuracy: 0.9969 - val_loss: 0.0360 - val_accuracy: 0.9908
Epoch 3/3
1688/1688 [==============================] - 9s 5ms/step - loss: 0.0088 - accuracy: 0.9979 - val_loss: 0.0347 - val_accuracy: 0.9915


## Sparsity 확인

In [ ]:
# 각 layer 별 sparsity를 확인 하기 위한 함수
def print_model_weights_sparsity(model):
    for layer in model.layers:
        if isinstance(layer, keras.layers.Wrapper):
            weights = layer.trainable_weights
        else:
            weights = layer.weights
        for weight in weights:
            if "kernel" not in weight.name or "centroid" in weight.name:    # 추후 clustered model 고려
                continue
            weight_size = weight.numpy().size
            zero_num = np.count_nonzero(weight == 0)
            print(
                f"{weight.name}: {zero_num/weight_size:.2%} sparsity ",
                f"({zero_num}/{weight_size})"
            )

In [ ]:
stripped_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

# pruned model의 sparsity 확인
print_model_weights_sparsity(stripped_pruned_model)

conv2d/kernel:0: 50.00% sparsity  (144/288)
conv2d_1/kernel:0: 50.00% sparsity  (2304/4608)
dense/kernel:0: 50.00% sparsity  (25600/51200)
dense_1/kernel:0: 50.00% sparsity  (640/1280)


## 단순 Clustering 진행
* clustering fine-tune 과정에서 sparsity 파괴 예정

In [ ]:
stripped_pruned_model_copy = keras.models.clone_model(stripped_pruned_model) # stripped_pruned_model_copy : 단순 clustering 진행 시 사용
stripped_pruned_model_copy.set_weights(stripped_pruned_model.get_weights())

In [ ]:
# Clustering : clustering fine-tune 과정에서 sparsity는 파괴 예정
clustering_params = {
  'number_of_clusters': 8,
  'cluster_centroids_init': tfmot.clustering.keras.CentroidInitialization.KMEANS_PLUS_PLUS
}

clustered_model = tfmot.clustering.keras.cluster_weights(stripped_pruned_model_copy, **clustering_params)

clustered_model.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

print('Train clustering model:')
clustered_model.fit(train_images, train_labels,epochs=3, validation_split=0.1)

stripped_pruned_model.save("stripped_pruned_model_clustered.h5")

Train clustering model:
Epoch 1/3
1688/1688 [==============================] - 12s 5ms/step - loss: 0.0079 - accuracy: 0.9974 - val_loss: 0.0472 - val_accuracy: 0.9892
Epoch 2/3
1688/1688 [==============================] - 10s 6ms/step - loss: 0.0077 - accuracy: 0.9975 - val_loss: 0.0497 - val_accuracy: 0.9895
Epoch 3/3
1688/1688 [==============================] - 10s 6ms/step - loss: 0.0102 - accuracy: 0.9967 - val_loss: 0.0471 - val_accuracy: 0.9890


/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


## Sparty preserving clustering 진행
* sparsity를 보존하며 clustering 진행

In [ ]:
# Sparsity preserving clustering

clustering_params = {
  'number_of_clusters': 8,
  'cluster_centroids_init': tfmot.clustering.keras.CentroidInitialization.KMEANS_PLUS_PLUS,
  'preserve_sparsity': True     # sparsity 보존
}

sparsity_clustered_model = tfmot.clustering.keras.cluster_weights(stripped_pruned_model, **clustering_params)

sparsity_clustered_model.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

print('Train sparsity preserving clustering model:')
sparsity_clustered_model.fit(train_images, train_labels,epochs=3, validation_split=0.1)

Train sparsity preserving clustering model:
Epoch 1/3
1688/1688 [==============================] - 17s 7ms/step - loss: 0.0070 - accuracy: 0.9976 - val_loss: 0.0402 - val_accuracy: 0.9912
Epoch 2/3
1688/1688 [==============================] - 11s 7ms/step - loss: 0.0057 - accuracy: 0.9982 - val_loss: 0.0558 - val_accuracy: 0.9897
Epoch 3/3
1688/1688 [==============================] - 11s 7ms/step - loss: 0.0057 - accuracy: 0.9981 - val_loss: 0.0447 - val_accuracy: 0.9917


## Sparsity 확인 : Clustered model VS Sparsity preserved clustered model

In [ ]:
print("Sparsity preserved clustered Model sparsity:")
print_model_weights_sparsity(sparsity_clustered_model)

print("\nClustered Model sparsity:")
print_model_weights_sparsity(clustered_model)

Sparsity preserved clustered Model sparsity:
conv2d/kernel:0: 50.00% sparsity  (144/288)
conv2d_1/kernel:0: 50.00% sparsity  (2304/4608)
dense/kernel:0: 50.00% sparsity  (25600/51200)
dense_1/kernel:0: 50.00% sparsity  (640/1280)

Clustered Model sparsity:
conv2d/kernel:0: 0.00% sparsity  (0/288)
conv2d_1/kernel:0: 0.00% sparsity  (0/4608)
dense/kernel:0: 2.80% sparsity  (1435/51200)
dense_1/kernel:0: 0.00% sparsity  (0/1280)


## Accuracy 비교 : Baseline model VS Sparsity preserving clustered model

In [ ]:
_, sparsity_clustered_model_accuracy = sparsity_clustered_model.evaluate(
  test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)
print('Sparsity preserving clustered test accuracy:', sparsity_clustered_model_accuracy)

Baseline test accuracy: 0.9904000163078308
Sparsity preserving clustered test accuracy: 0.9905999898910522


## Sparsity and Cluster preserving QAT

In [ ]:
stripped_sparsity_clustered_model = tfmot.clustering.keras.strip_clustering(sparsity_clustered_model)

quant_aware_annotate_model = tfmot.quantization.keras.quantize_annotate_model(
    stripped_sparsity_clustered_model
)

pcqat_model = tfmot.quantization.keras.quantize_apply(
    quant_aware_annotate_model,
    tfmot.experimental.combine.Default8BitClusterPreserveQuantizeScheme(preserve_sparsity=True)
)

pcqat_model.compile(optimizer='adam',
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

print('Train pcqat model:')
pcqat_model.fit(train_images, train_labels, batch_size=128, epochs=1, validation_split=0.1)

Train pcqat model:


422/422 [==============================] - 9s 11ms/step - loss: 0.0025 - accuracy: 0.9994 - val_loss: 0.0460 - val_accuracy: 0.9918


In [ ]:
stripped_sparsity_clustered_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0         
                                                                 
 dense (Dense)               (None, 128)               5

In [ ]:
def print_model_weight_clusters(model):
    for layer in model.layers:
        if isinstance(layer, keras.layers.Wrapper):
            weights = layer.trainable_weights
        else:
            weights = layer.weights
        for weight in weights:
            # ignore auxiliary quantization weights
            if "quantize_layer" in weight.name:
                continue
            if "kernel" in weight.name:
                unique_count = len(np.unique(weight))
                print(
                    f"{layer.name}/{weight.name}: {unique_count} clusters "
                )

In [ ]:
print("\nPCQAT Model clusters:")
print_model_weight_clusters(pcqat_model)
print("\nPCQAT Model sparsity:")
print_model_weights_sparsity(pcqat_model)


PCQAT Model clusters:
quant_conv2d/conv2d/kernel:0: 8 clusters 
quant_conv2d_1/conv2d_1/kernel:0: 8 clusters 
quant_dense/dense/kernel:0: 8 clusters 
quant_dense_1/dense_1/kernel:0: 8 clusters 

PCQAT Model sparsity:
conv2d/kernel:0: 53.47% sparsity  (154/288)
conv2d_1/kernel:0: 54.51% sparsity  (2512/4608)
dense/kernel:0: 59.18% sparsity  (30300/51200)
dense_1/kernel:0: 52.89% sparsity  (677/1280)


## LiteRT 모델로 변환 (PCQAT)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(pcqat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [ ]:
tflite_pcqat_file = save_dir + 'mnist_pcqat.tflite'
open(tflite_pcqat_file, 'wb').write(tflite_model)

63888

## LiteRT 설치 및 Interpreter 로딩

In [ ]:
!pip install ai-edge-litert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 84.0 MB/s eta 0:00:00


In [ ]:
from ai_edge_litert.interpreter import Interpreter

## interpreter 생성 (PCQAT)

In [ ]:
interpreter_pcqat = Interpreter(model_path=str(tflite_pcqat_file))
interpreter_pcqat.allocate_tensors()

## input/output dtype 확인 (PQAT)

In [ ]:
input_dtype = interpreter_pcqat.get_input_details()[0]['dtype']
output_dtype = interpreter_pcqat.get_output_details()[0]['dtype']

print("input dtype : {}".format(input_dtype))
print("output dtype : {}".format(output_dtype))

input dtype : <class 'numpy.float32'>
output dtype : <class 'numpy.float32'>


## Test data 기반 accuracy 평가

In [ ]:
def eval_model(interpreter):
  input_details = interpreter.get_input_details()[0]
  output_details = interpreter.get_output_details()[0]
  input_index = input_details["index"]
  output_index = output_details["index"]

  # Run predictions on every image in the "test" dataset.
  prediction_digits = []
  for i, test_image in enumerate(test_images):
    test_image = np.expand_dims(test_image, axis=0).astype(input_details["dtype"])
    interpreter.set_tensor(input_index, test_image)

    # Run inference.
    interpreter.invoke()

    # Post-processing: remove batch dimension and find the digit with highest
    # probability.
    output = interpreter.get_tensor(output_index)
    digit = np.argmax(output)
    prediction_digits.append(digit)

  # Compare prediction results with ground truth labels to calculate accuracy.
  prediction_digits = np.array(prediction_digits)
  accuracy = (prediction_digits == test_labels).mean()
  return accuracy

In [ ]:
pcqat_test_accuracy = eval_model(interpreter_pcqat)

print('PCQAT test_accuracy:', pcqat_test_accuracy)
print('Baseline model test_accuracy :', baseline_model_accuracy)

PCQAT test_accuracy: 0.992
Baseline model test_accuracy : 0.9904000163078308
